
## Install & Imports



In [1]:
!pip install pypdf sentence-transformers faiss-cpu
!pip install -q -U google-generativeai pypdf sentence-transformers faiss-cpu
!pip install -q -U semantic-chunkers
!pip install -U langchain langchain-community
!pip install google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 42.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import nltk
import pandas as pd
import numpy as np
from pypdf import PdfReader
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
nltk.download('punkt')
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

# Load PDFs

In [4]:
file_names = [
    "AI_Ethics_UNESCO_Detailed.pdf",
    "aftermath_word_war_2.pdf",
    "World_War_II_Comprehensive_Analysis.pdf",
    "World_War_I_Comprehensive_Report.pdf",
    "NIPS-2017-attention-is-all-you-need-Paper.pdf"
]

pdf_data = {}

def load_pdfs(files):
    for file in files:
        reader = PdfReader(file)
        text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
        pdf_data[file] = text

load_pdfs(file_names)

print("Loaded PDFs:", len(pdf_data))

Loaded PDFs: 5


# Clean Text

In [5]:
def clean_text(text):
    text = re.sub(r'-\s*\n\s*', '', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\b\d{1,3}\b(?=\s)', '', text)
    return text.strip()

for k in pdf_data:
    pdf_data[k] = clean_text(pdf_data[k])

# Load Embedding Model

In [6]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Semantic Chunking

In [7]:
def semantic_chunking(text, threshold=0.55, max_words=200):
    sentences = sent_tokenize(text)
    if not sentences:
        return [], []

    sentence_emb = embed_model.encode(sentences)

    chunks, metadata = [], []
    current_chunk, current_emb = [], []

    chunk_id = 0

    for i, sent in enumerate(sentences):
        emb = sentence_emb[i]

        if not current_chunk:
            current_chunk.append(sent)
            current_emb.append(emb)
            continue

        sim = cosine_similarity([current_emb[-1]], [emb])[0][0]

        if sim < threshold or len(" ".join(current_chunk).split()) > max_words:
            text_chunk = " ".join(current_chunk)

            chunks.append(text_chunk)
            metadata.append({
                "id": str(uuid.uuid4()),
                "chunk_index": chunk_id,
                "length": len(text_chunk),
                "preview": text_chunk[:120]
            })

            chunk_id += 1
            current_chunk = [sent]
            current_emb = [emb]

        else:
            current_chunk.append(sent)
            current_emb.append(emb)

    if current_chunk:
        text_chunk = " ".join(current_chunk)
        chunks.append(text_chunk)
        metadata.append({
            "id": str(uuid.uuid4()),
            "chunk_index": chunk_id,
            "length": len(text_chunk),
            "preview": text_chunk[:120]
        })

    return chunks, metadata

# Chunk All PDFs

In [8]:
all_chunks = []
chunk_metadata = []
pdf_chunks = {}

for file, text in pdf_data.items():
    chunks, meta = semantic_chunking(text)

    pdf_chunks[file] = chunks

    start = len(all_chunks)
    all_chunks.extend(chunks)

    for i, m in enumerate(meta):
        m["source"] = file
        m["global_chunk_index"] = start + i
        chunk_metadata.append(m)

print("Total chunks:", len(all_chunks))

Total chunks: 467


# Embeddings

In [9]:
embeddings = embed_model.encode(
    all_chunks,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Embedding shape: (467, 384)


# Build FAISS Index

In [11]:
import faiss

faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS index size:", index.ntotal)

FAISS index size: 467


# Save Index + Data

In [12]:
faiss.write_index(index, "faiss_index.bin")

with open("data.pkl", "wb") as f:
    pickle.dump({
        "chunks": all_chunks,
        "metadata": chunk_metadata
    }, f)

print("Saved FAISS index and metadata.")

Saved FAISS index and metadata.


# Initialize Gemini Model

In [41]:
from google.colab import userdata
import google.generativeai as genai

api_key = userdata.get("GOOGLE_API_KEY")
genai.configure(api_key=api_key)

MODEL_NAME = "models/gemini-2.5-flash"

llm = genai.GenerativeModel(MODEL_NAME)

print("Using model:", MODEL_NAME)

Using model: models/gemini-2.5-flash


# Query Embedding

In [42]:
def embed_query(query):
    return embed_model.encode([query]).astype("float32")

# Retrieve Top-K Chunks from FAISS

In [43]:
def retrieve_chunks(query, k=3):
    query_vec = embed_query(query)
    faiss.normalize_L2(query_vec)

    scores, indices = index.search(query_vec, k)

    chunks = []
    metas = []

    for i in indices[0]:
        if i != -1:
            chunks.append(all_chunks[i])
            metas.append(chunk_metadata[i])  # important fix

    return chunks, metas

# MODIFY CONTEXT BUILDER

In [44]:
def build_context(chunks, metas):
    context = ""

    for chunk, meta in zip(chunks, metas):
        context += f"""
[Source: {meta['source']} | Chunk {meta['chunk_index']}]
{chunk}

"""

    return context

# Build Context for Gemini

In [45]:
def build_context(chunks, metas):
    context = ""

    for chunk, meta in zip(chunks, metas):
        context += f"""
[Source: {meta['source']} | Chunk {meta['chunk_index']}]
{chunk}

"""

    return context

# Gemini RAG Answer Function

In [49]:
def ask_bot(query):
    chunks, metas = retrieve_chunks(query, k=3)
    context = build_context(chunks, metas)

    prompt = f"""
You are an expert exam tutor.

You must answer using ONLY the provided context.

STRICT RULES:
- Do NOT use outside knowledge.
- Do NOT introduce new facts not present in the context.
- You MAY rephrase, organize, and slightly elaborate ONLY based on the given context.
- You MAY add headings, explanations, and structure to improve clarity.
- If something is missing in context, do not invent it.

OUTPUT FORMAT:
1. Definition (clear and simple)
2. Explanation (well-structured, slightly expanded from context)
3. Key Points (well-organized bullets)
4. Sources

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.generate_content(prompt)
    return response.text

# Testing The Chatbot

In [50]:
query = "Who was hitler?"
print(ask_bot(query))

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2100.51ms


Here is information about Hitler, based solely on the provided context:

### Definition

Adolf Hitler was an individual who, alongside the Nazi Party, rose to power in Germany and later committed suicide, preceding Germany's unconditional surrender.

### Explanation

Adolf Hitler came to prominence by leading the Nazi Party. This party gained power in a political vacuum, which emerged after Germany was left economically devastated and politically humiliated by the Treaty of Versailles following World War I. Hitler and the Nazi Party's platform was based on principles of nationalism and expansionism. His death, by suicide, occurred before Germany's unconditional surrender on May 8, 1945, a day known as V-E Day.

### Key Points

*   **Rise to Power:** Adolf Hitler, along with the Nazi Party, rose to power.
*   **Political Platform:** Their platform was founded on nationalism and expansionism.
*   **Circumstances of Death:** He committed suicide.
*   **Impact of Death:** His suicide prece